# SUNDOR AIRLINE OPERATIONS ANALYTICS PROJECT

Project Overview

This project simulates a realistic airline operations dataset for Sundor Airlines between 2020 and 2025.

The objective of the project is to create a business-oriented analytical environment that reflects the major operational, commercial, and external factors that influence airline performance. The dataset is designed for use in Power BI dashboards, SQL analysis, forecasting models, KPI monitoring, and executive-level reporting.

Instead of generating random airline data, the project was built around realistic business assumptions and industry drivers that affect demand, revenue, and operational performance.

Dataset Structure

The project follows a star-schema data model commonly used in enterprise data warehouses.

Dimension tables include:

• Date
• Route
• Aircraft
• Holidays
• Major Aviation Events

Fact tables include:

• Daily Route Operations
• Fuel Price Index
• Security Risk Index
• Competition Index

The final operational dataset contains one record per route per day across all Sundor destinations.

Demand Model

Passenger demand is generated using a multi-factor demand model.

The model considers:

• Base route demand
• Destination popularity
• Route tier classification
• Seasonal demand patterns
• Israeli holiday periods
• Security-related events
• Competition intensity
• Daily market fluctuations

Seasonality

The model reflects common leisure travel patterns.

Examples include:

• Summer demand increase (June-August)
• Shoulder season reductions
• Holiday travel peaks

Holiday Effects

Real Israeli holiday dates are incorporated into the model.

Examples:

• Passover
• Sukkot
• Hanukkah

Holiday periods increase travel demand and influence ticket pricing.

Security Impact

A Security Risk Index was created to simulate the impact of significant events on aviation demand.

Examples include:

• COVID-19 Pandemic
• May 2021 Conflict
• October 7 War
• Iranian Missile Attacks
• Regional Security Escalations

Competition Impact

A Competition Index was included to represent market pressure from competing airlines and low-cost carriers.

Higher competition reduces pricing power and impacts demand.

Passenger & Capacity Model

The model generates passenger volumes based on route demand and available seat capacity.

Passengers are constrained by aircraft capacity and scheduled operations.

Load Factor is calculated as:

Passengers ÷ Seats Offered

The simulation produces realistic airline utilization levels with an average load factor close to 80%.

Revenue Model

Revenue is generated through two primary streams:

1. Ticket Revenue

Average fares are dynamically calculated using:

• Route pricing ranges
• Demand levels
• Load factors
• Seasonality
• Holiday periods

2. Ancillary Revenue

Additional revenue is generated from:

• Checked baggage
• Seat selection
• Priority boarding
• Onboard purchases

Longer routes generally produce higher ancillary revenue per passenger.

Business Objective

The purpose of this project is not to predict actual Sundor financial performance.

Instead, the objective is to demonstrate how operational data, external events, market conditions, and business drivers can be integrated into a realistic airline analytics model.

The dataset supports:

• Route performance analysis
• Executive KPI reporting
• Trend analysis
• Forecasting
• Event impact analysis
• Revenue optimization analysis
• Power BI dashboard development
• SQL-based analytical projects

Key Takeaway

This project demonstrates a business-focused approach to data analytics by combining airline operations, market dynamics, seasonality, competition, and external events into a unified analytical model.

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

OUTPUT_FOLDER = "../Data/Generated/"

In [3]:
dim_route = pd.read_csv(
    OUTPUT_FOLDER + "dim_route.csv"
)

dim_aircraft = pd.read_csv(
    OUTPUT_FOLDER + "dim_aircraft.csv"
)

dim_date = pd.read_csv(
    OUTPUT_FOLDER + "dim_date.csv"
)

dim_holiday = pd.read_csv(
    OUTPUT_FOLDER + "dim_holiday.csv"
)

fuel = pd.read_csv(
    OUTPUT_FOLDER + "fact_fuel_index.csv"
)

security = pd.read_csv(
    OUTPUT_FOLDER + "fact_security_risk.csv"
)

competition = pd.read_csv(
    OUTPUT_FOLDER + "fact_competition.csv"
)

print("Tables loaded successfully")

Tables loaded successfully


In [4]:
dim_date["Date"] = pd.to_datetime(
    dim_date["Date"]
)

dim_holiday["Date"] = pd.to_datetime(
    dim_holiday["Date"]
)

fuel["Date"] = pd.to_datetime(
    fuel["Date"]
)

security["Date"] = pd.to_datetime(
    security["Date"]
)

competition["Date"] = pd.to_datetime(
    competition["Date"]
)

In [5]:
fact = dim_date.merge(
    dim_route,
    how="cross"
)

print("Rows:", len(fact))

Rows: 56992


In [6]:
fact = fact.merge(
    fuel,
    on="Date",
    how="left"
)

In [7]:
fact = fact.merge(
    security,
    on="Date",
    how="left"
)

In [8]:
fact = fact.merge(
    dim_holiday,
    on="Date",
    how="left"
)

fact["Demand_Impact"] = (
    fact["Demand_Impact"]
    .fillna(0)
)

In [9]:
fact = fact.merge(
    dim_aircraft[
        [
            "Aircraft_Type",
            "Seats",
            "Fuel_Burn_Index"
        ]
    ],
    on="Aircraft_Type",
    how="left"
)

In [10]:
fact["Demand_Index"] = 100

In [11]:
fact.loc[
    fact["Month"].isin([6, 7, 8]),
    "Demand_Index"
] *= 1.20

fact.loc[
    fact["Month"].isin([10, 11]),
    "Demand_Index"
] *= 0.90

In [12]:
fact["Demand_Index"] *= (
    1 + fact["Demand_Impact"]
)

In [13]:
fact["Demand_Index"] *= (
    1 - (
        fact["Security_Risk_Index"] * 0.03
    )
)

In [14]:
fact = fact.merge(
    competition,
    on="Date",
    how="left"
)

In [15]:
print("Competition_Index" in fact.columns)

True


In [16]:
fact["Demand_Index"] *= (
    1 + (
        (100 - fact["Competition_Index"])
        / 300
    )
)

In [17]:
fact["Demand_Index"] *= np.random.normal(
    1,
    0.08,
    len(fact)
)

In [18]:
fact["Demand_Index"] = (
    fact["Demand_Index"]
    .clip(lower=20, upper=250)
)

In [19]:
print(
    "Min Demand:",
    round(fact["Demand_Index"].min(), 2)
)

print(
    "Max Demand:",
    round(fact["Demand_Index"].max(), 2)
)

print(
    "Average Demand:",
    round(fact["Demand_Index"].mean(), 2)
)

Min Demand: 44.3
Max Demand: 162.5
Average Demand: 101.26


In [20]:
fact["Flights_Operated"] = (
    fact["Base_Daily_Flights"]
)

fact.loc[
    (fact["Seasonal_Route"] == True)
    &
    (fact["Month"].isin([6, 7, 8])),
    "Flights_Operated"
] += 1

In [21]:
fact["Seats_Offered"] = (
    fact["Flights_Operated"]
    *
    fact["Seats"]
)

In [22]:
fact["Route_Demand_Multiplier"] = np.where(
    fact["Route_Tier"] == 1,
    3.3,
    np.where(
        fact["Route_Tier"] == 2,
        2.8,
        2.2
    )
)

In [23]:
fact["Expected_Passengers"] = (
    fact["Base_Daily_Demand"]
    *
    fact["Route_Demand_Multiplier"]
    *
    (fact["Demand_Index"] / 100)
)

In [24]:
fact["Passengers"] = (
    fact["Expected_Passengers"]
    *
    np.random.normal(
        1,
        0.05,
        len(fact)
    )
)

fact["Passengers"] = (
    fact["Passengers"]
    .round()
)

In [25]:
fact["Passengers"] = np.minimum(
    fact["Passengers"],
    fact["Seats_Offered"]
)

In [26]:
fact["Load_Factor"] = (
    fact["Passengers"]
    /
    fact["Seats_Offered"]
)

fact["Load_Factor"] = (
    fact["Load_Factor"]
    .clip(
        lower=0.20,
        upper=0.98
    )
)

In [27]:
print(
    "Average Load Factor:",
    round(
        fact["Load_Factor"].mean(),
        3
    )
)

print(
    "Min Load Factor:",
    round(
        fact["Load_Factor"].min(),
        3
    )
)

print(
    "Max Load Factor:",
    round(
        fact["Load_Factor"].max(),
        3
    )
)

Average Load Factor: 0.798
Min Load Factor: 0.283
Max Load Factor: 0.98


# Revenue & Cost Engines

In [28]:
# Calculate Fare Position, higher demand = higher fares
fact["Fare_Position"] = (
    0.4
    +
    (fact["Load_Factor"] * 0.6)
)

In [41]:
fact["Avg_Fare_USD"] = (
    fact["Fare_Low_USD"]
    +
    (
        fact["Fare_High_USD"]
        -
        fact["Fare_Low_USD"]
    )
    *
    fact["Fare_Position"]
)

# Revenue calibration
fact["Avg_Fare_USD"] *= 0.85

In [42]:
fact.loc[
    fact["Month"].isin([6,7,8]),
    "Avg_Fare_USD"
] *= 1.10

In [43]:
fact.loc[
    fact["Holiday_Name"].notna(),
    "Avg_Fare_USD"
] *= 1.08

In [44]:
# Fare noise
fact["Avg_Fare_USD"] *= np.random.normal(
    1,
    0.05,
    len(fact)
)

In [45]:
# Fare validation
print(
    "Average Fare:",
    round(
        fact["Avg_Fare_USD"].mean(),
        2
    )
)

print(
    "Min Fare:",
    round(
        fact["Avg_Fare_USD"].min(),
        2
    )
)

print(
    "Max Fare:",
    round(
        fact["Avg_Fare_USD"].max(),
        2
    )
)

Average Fare: 226.18
Min Fare: 111.18
Max Fare: 429.8


In [46]:
# Ticket Revenue
fact["Ticket_Revenue_USD"] = (
    fact["Passengers"]
    *
    fact["Avg_Fare_USD"]
)

In [47]:
# Ancillary Revenue per Pax, higher for longer routes = Short-haul passengers spend less.
fact["Ancillary_Per_Pax"] = np.where(
    fact["Distance_KM"] < 1000,
    np.random.uniform(18,25,len(fact)),
    np.where(
        fact["Distance_KM"] < 2500,
        np.random.uniform(22,32,len(fact)),
        np.random.uniform(28,45,len(fact))
    )
)

In [48]:
# Ancillary Revenue
fact["Ancillary_Revenue_USD"] = (
    fact["Passengers"]
    *
    fact["Ancillary_Per_Pax"]
)

In [49]:
# Total Revenue 
fact["Total_Revenue_USD"] = (
    fact["Ticket_Revenue_USD"]
    +
    fact["Ancillary_Revenue_USD"]
)

In [50]:
# Revenue Validation
print(
    fact[
        [
            "City",
            "Passengers",
            "Avg_Fare_USD",
            "Ticket_Revenue_USD",
            "Ancillary_Revenue_USD",
            "Total_Revenue_USD"
        ]
    ].sample(
        10,
        random_state=42
    )
)

            City  Passengers  Avg_Fare_USD  Ticket_Revenue_USD  \
7984   Heraklion       339.0    197.245172        66866.113312   
12428     Athens       660.0    207.079935       136672.757117   
36920     Athens       457.0    196.848500        89959.764301   
28422      Sitia       118.0    185.773018        21921.216119   
41841   Sardinia       152.0    249.690627        37952.975305   
49351  Kefalonia       126.0    193.282097        24353.544228   
14261  Dubrovnik       243.0    239.154705        58114.593286   
55642  Heraklion       300.0    193.084630        57925.389005   
33603      Basel       170.0    266.080253        45233.642988   
23577     Warsaw       528.0    301.359394       159117.759910   

       Ancillary_Revenue_USD  Total_Revenue_USD  
7984             8175.827217       75041.940529  
12428           20039.363163      156712.120280  
36920           12276.249747      102236.014048  
28422            3528.785392       25450.001511  
41841            4310.1

In [51]:
# Revenue Summary
print(
    "Total Revenue:",
    round(
        fact["Total_Revenue_USD"].sum()/1_000_000,
        2
    ),
    "Million USD"
)

print(
    "Average Daily Revenue:",
    round(
        fact["Total_Revenue_USD"].mean(),
        2
    )
)

Total Revenue: 4138.42 Million USD
Average Daily Revenue: 72614.1


In [52]:
fact.to_csv(
    OUTPUT_FOLDER + "fact_operations_stage1.csv",
    index=False
)

print("fact_operations_stage1.csv saved")

fact_operations_stage1.csv saved
